# sphinxcontrib_devhelp

sphinxcontrib-devhelp 2.0.0 is Sphinx's `devhelp` builder: HTML pages plus a gzipped `<project>.devhelp.gz` book index
for GNOME Devhelp. These cells build a book, read the index's chapters and keywords, and check the registration.

In [ ]:
# Example 1: a Devhelp build writes the HTML pages and the compressed book index.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nIntro with *emphasis*.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'devhelp', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
assert (out / 'Demo.devhelp.gz').exists() and (out / 'page.html').exists()

In [ ]:
# Example 2: the book index is gzipped XML naming the book and its chapters.
import gzip
import xml.etree.ElementTree as ET
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nIntro with *emphasis*.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'devhelp', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
root = ET.fromstring(gzip.decompress((out / 'Demo.devhelp.gz').read_bytes()))
print(root.tag, root.attrib)
assert root.tag == 'book' and root.attrib['name'] == 'Demo' and root.attrib['link'] == 'index.html'
assert [s.attrib['link'] for s in root.iter('sub')] == ['page.html']

In [ ]:
# Example 3: index entries become Devhelp <function> keywords that point into the pages.
import gzip
import xml.etree.ElementTree as ET
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nIntro with *emphasis*.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'devhelp', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
root = ET.fromstring(gzip.decompress((out / 'Demo.devhelp.gz').read_bytes()))
keywords = [(k.attrib['name'], k.attrib['link']) for k in root.iter('function')]
print(keywords)
assert any(name == 'widgets' and link.startswith('page.html') for name, link in keywords)

In [ ]:
# Example 4: the builder's registration.
import sphinxcontrib.devhelp as devhelp
from sphinxcontrib.devhelp import DevhelpBuilder

print(devhelp.__version__, DevhelpBuilder.name)
assert devhelp.__version__ == "2.0.0"
assert DevhelpBuilder.name == "devhelp" and DevhelpBuilder.embedded is True